# 02 — Features, entraînement et évaluation

Classification des 3 gestes MPU6050 (`vertical`, `horizontal`, `circle`) à partir de features **par essai**.

**Important :** le split train/val/test se fait par `trial_id` (GroupShuffleSplit) pour éviter toute fuite entre échantillons du même mouvement.

**Répertoire de travail :** racine du dépôt.

```bash
jupyter notebook notebooks/02_train_eval.ipynb
```

Placez d'abord les CSV dans `data/raw/` :
`vertical.csv`, `horizontal.csv`, `circle.csv`.


In [ ]:
from pathlib import Path
import sys
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "python"))

from gesture_dataset import (
    LABELS,
    extract_trial_features,
    feature_matrix,
    load_gesture_csvs,
)

RAW_DIR = ROOT / "data" / "raw"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
RANDOM_STATE = 42
print("ROOT =", ROOT)


## Chargement + extraction de features


In [ ]:
df = load_gesture_csvs(RAW_DIR)
features = extract_trial_features(df)
print(f"Essais : {len(features)}")
display(features["label"].value_counts().reindex(LABELS))
features.head()


In [ ]:
X, y, groups = feature_matrix(features)
print(f"X shape = {X.shape}")
print("Features :", list(X.columns))


## Split par essai (sans fuite)

60 % train / 20 % val / 20 % test — groupes = `trial_id`.


In [ ]:
# 1er split : train+val vs test (80/20)
gss_outer = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_val_idx, test_idx = next(gss_outer.split(X, y, groups))

X_train_val, y_train_val = X.iloc[train_val_idx], y.iloc[train_val_idx]
groups_train_val = groups.iloc[train_val_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

# 2e split : train vs val (75/25 du train_val ≈ 60/20 global)
gss_inner = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, val_idx = next(gss_inner.split(X_train_val, y_train_val, groups_train_val))

X_train, y_train = X_train_val.iloc[train_idx], y_train_val.iloc[train_idx]
X_val, y_val = X_train_val.iloc[val_idx], y_train_val.iloc[val_idx]

def _balance(series, name):
    print(f"\n{name} (n={len(series)})")
    print(series.value_counts().reindex(LABELS).fillna(0).astype(int))

_balance(y_train, "train")
_balance(y_val, "val")
_balance(y_test, "test")

# Vérif : aucun trial_id partagé
assert set(groups.iloc[train_val_idx.tolist()]) & set(groups.iloc[test_idx.tolist()]) == set()
assert set(groups_train_val.iloc[train_idx.tolist()]) & set(groups_train_val.iloc[val_idx.tolist()]) == set()
print("\nOK — aucun trial_id en commun entre les splits.")


## Modèles candidats

- RandomForest (robuste, peu de scaling)
- SVM RBF (+ StandardScaler)
- Régression logistique multinomiale (+ StandardScaler)


In [ ]:
candidates = {
    "random_forest": Pipeline(
        [
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=200,
                    max_depth=None,
                    min_samples_leaf=2,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            )
        ]
    ),
    "svm_rbf": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "clf",
                SVC(
                    kernel="rbf",
                    C=2.0,
                    gamma="scale",
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "logreg": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "clf",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
}

val_rows = []
fitted = {}
for name, pipe in candidates.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)
    acc = accuracy_score(y_val, pred)
    f1 = f1_score(y_val, pred, average="macro", labels=list(LABELS))
    val_rows.append({"model": name, "val_accuracy": acc, "val_f1_macro": f1})
    fitted[name] = pipe
    print(f"{name:15s}  val_acc={acc:.3f}  val_f1_macro={f1:.3f}")

val_df = pd.DataFrame(val_rows).sort_values("val_f1_macro", ascending=False)
display(val_df)
best_name = val_df.iloc[0]["model"]
best_model = fitted[best_name]
print(f"\nMeilleur modèle (val) : {best_name}")


## Réentraînement train+val → évaluation test


In [ ]:
# Réentraîner le meilleur sur train+val
best_final = candidates[best_name]
best_final.fit(X_train_val, y_train_val)
y_pred = best_final.predict(X_test)

test_acc = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average="macro", labels=list(LABELS))
print(f"Test accuracy     : {test_acc:.3f}")
print(f"Test F1 macro     : {test_f1:.3f}")
print()
print(classification_report(y_test, y_pred, labels=list(LABELS), digits=3))


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=list(LABELS))
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=list(LABELS)).plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Matrice de confusion — {best_name}")
plt.tight_layout()
plt.show()


## Importance des features (si RandomForest)


In [ ]:
clf = best_final.named_steps.get("clf")
if hasattr(clf, "feature_importances_"):
    imp = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
    display(imp.head(15).to_frame("importance"))
    ax = imp.head(15).iloc[::-1].plot(kind="barh", figsize=(7, 5), color="#3d5a80")
    ax.set_title("Top 15 features")
    ax.set_xlabel("importance")
    plt.tight_layout()
    plt.show()
else:
    print(f"Pas d'importances natives pour {best_name}.")


## Sauvegarde de l'artefact


In [ ]:
model_path = ARTIFACTS / "gesture_classifier.joblib"
meta_path = ARTIFACTS / "gesture_classifier_meta.json"

joblib.dump(
    {
        "model": best_final,
        "feature_columns": list(X.columns),
        "labels": list(LABELS),
        "model_name": best_name,
    },
    model_path,
)

meta = {
    "model_name": best_name,
    "test_accuracy": float(test_acc),
    "test_f1_macro": float(test_f1),
    "n_train": int(len(y_train)),
    "n_val": int(len(y_val)),
    "n_test": int(len(y_test)),
    "n_features": int(X.shape[1]),
    "feature_columns": list(X.columns),
}
meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("Modèle :", model_path)
print("Meta   :", meta_path)
print(json.dumps(meta, indent=2))


## (Optionnel) Approche séquence brute

Pour un petit jeu de données (quelques centaines d'essais × 75 × 6), les features statistiques + RandomForest/SVM sont en général suffisantes. Un modèle de séquence (CNN 1D / LSTM) n'est justifié qu'avec plus de données et une validation croisée par groupes — hors scope de ce notebook de baseline.
